# Latest results: write → next read
Selected prn_1 mixtures and proj_1 address medians: training on days 1–3, selection by validation MAE on day 4, development evaluation on days 5–7. On prn_1, MAE is below even the lower bound on mean overwrite time, but the fraction satisfying error < overwrite does not reach 90%.

Below are verified metrics, model loading and an inference example, followed by the earlier usr_1/prxy_1 study. Kernel: `.venv/bin/python` after `uv sync --extra notebook`. The project root is detected automatically. Executing this notebook does not train models.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
from IPython.display import Markdown, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'read_prediction').is_dir() and (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from read_prediction.paths import REPORTS
from read_prediction.submission_overwrite import collect_results, summary_markdown
SUBMISSION = ROOT / 'submission'
latest_results = json.loads((SUBMISSION / 'overwrite-results.json').read_text())
assert latest_results == collect_results(), 'Submission must match the verified results'
display(Markdown(summary_markdown(latest_results)))

## Load selected predictors and run inference
prn_1: 25% log-L2 + 75% raw-L1, mixed after converting predictions to seconds. Separate model pairs are provided for same_start and range. proj_1: per-address median with a global-median fallback. Files and fingerprints are listed in submission/models.json.

Input: seven raw features — Offset in bytes, size in bytes, previous Read count, previous Write count, seconds since the last Read, seconds since the last Write, and the last completed Read-to-Read interval. Missing history is NaN. History uses the same starting Offset and is available before the current Write for both rules. The predictor applies log1p internally.

In [ ]:
from submission.predict import SelectedPredictor
predictor = SelectedPredictor(SUBMISSION)
print(json.dumps(predictor.metadata['predictors'], indent=2, ensure_ascii=False))

# Illustrative features, not an evaluation sample from the trace.
raw_features = [[4096, 512, 0, 0, np.nan, np.nan, np.nan]]
for workload in ('prn_1', 'proj_1'):
    for rule in ('same_start', 'range'):
        seconds = predictor.predict_seconds(workload, rule, raw_features)
        assert np.isfinite(seconds).all() and (seconds >= 0).all()
        print(workload, rule, 'predicted seconds to Read:', seconds)

## Selected predictors: validation and individual future days
Days 5–7 have already been inspected: this is development evaluation, not a new holdout. Mean overwrite time is represented by a lower bound when overwrite times are unknown. The pass fraction accounts for unresolved cases through lower and upper bounds; MAE applies only to writes with an observed Read.

In [ ]:
table = ['| Workload / rule | Window | MAE, s | Mean overwrite ≥, s | MAE < mean | error < overwrite | ≥90% |', '|---|---|---:|---:|---|---:|---|']
mean_status = {'passed': 'proven', 'failed': 'no', 'undetermined': 'undetermined'}
pass_status = {'passed': 'yes', 'failed': 'no', 'undetermined': 'undetermined'}
for row in latest_results['selected_results']:
    metric = row['observed_read']
    table.append(f"| {row['workload']} / {row['rule']} | {row['window']} | {metric['mae_seconds']:.2f} | {metric['mean_overwrite_lower_seconds']:.2f} | {mean_status[metric['status']]} | {row['pass_lower']:.2%}–{row['pass_upper']:.2%} | {pass_status[row['ninety_percent_status']]} |")
display(Markdown('\n'.join(table)))

## Earlier study: usr_1 / prxy_1
The following tables describe different workloads and earlier models: day 1 training and validation selection, followed by frozen evaluation on days 2–4. Day 2 of prxy_1 reproduces an already evaluated period.

In [ ]:
RESULTS = ROOT / 'results' / 'final-study'
assert (RESULTS / 'verification.json').exists(), 'Earlier study results are required'
display(Markdown((REPORTS / 'summary.en.md').read_text()))

## Detailed report
Full English report: `reports/report.en.md`. English summary: `reports/summary.en.md`. All earlier-study metrics: `reports/metrics.csv`. Predictions: `results/final-study/<workload>/day<2|3|4>.predictions.npz`.

Metrics apply to writes with a Read before the end of their own day. Unknown intervals are not declared infinite or zero.

In [ ]:
for workload in ('usr_1', 'prxy_1'):
    record = json.loads((RESULTS / workload / 'results.json').read_text())
    print(workload, 'selected:', record['frozen']['selection']['method'])
    for day, result in record['days'].items():
        baseline = result['metrics']['address_median']['mae_seconds']
        selected = result['metrics']['selected']['mae_seconds']
        print(f'  Day {day}: median MAE={baseline:.2f}s, selected MAE={selected:.2f}s; {result["evaluation_role"]}')
print(json.dumps(json.loads((RESULTS / 'verification.json').read_text()), indent=2))

## Reproduction
Rebuild the portable model bundle: `.venv/bin/python -m read_prediction.submission`. Verify sources, 2048 saved predictions and portability: `.venv/bin/python -m read_prediction.checks.submission`. Verify mean-overwrite calculations: `.venv/bin/python -m read_prediction.checks.mean_overwrite`. Test standalone training: `.venv/bin/python -m read_prediction.checks.training_notebook`.

Earlier study: `uv run python -m read_prediction.checks.study`. Generate earlier-study reports and check this notebook: `uv run --extra notebook python -m read_prediction.reporting --check-notebook`. Training commands for a clean output directory are documented in report.en.md. Days 5–7 are development data; no new independent holdout is claimed here.